# CAD Retrieval Evaluation (Manufacturing-Aware)

This notebook evaluates the unsupervised CAD retrieval model.
Since the MFCAD dataset provides face-level machining feature labels, we construct a **Feature Histogram (Bag of Features)** for each CAD model. 

Two models are considered a "match" if the cosine similarity between their feature histograms exceeds a defined threshold (e.g., 0.95).

In [1]:
import os
import pickle
import numpy as np
import faiss
from pathlib import Path
from tqdm.notebook import tqdm

# ==========================================
# 1. Define Paths & Parameters
# ==========================================
PROJECT_ROOT = Path("..") 
FAISS_INDEX_PATH = PROJECT_ROOT / "checkpoints" / "faiss_index.bin"
METADATA_PATH = PROJECT_ROOT / "checkpoints" / "metadata.pkl"
LABEL_DIR = PROJECT_ROOT / "data" / "raw_step" / "mfcad_label"

NUM_CLASSES = 16 # Total machining feature types in MFCAD
MATCH_THRESHOLD = 0.95 # Cosine similarity threshold to consider models "similar"

print(f"Index exists: {FAISS_INDEX_PATH.exists()}")
print(f"Labels exist: {LABEL_DIR.exists()}")

Index exists: True
Labels exist: True


## Load Database & Generate Signatures
We load the index and generate a 16-D histogram for each CAD model based on its `.face_truth` file.

In [2]:
# ==========================================
# 2. Load FAISS Index & Metadata
# ==========================================
with open(METADATA_PATH, 'rb') as f:
    metadata = pickle.load(f)

index = faiss.read_index(str(FAISS_INDEX_PATH))
num_vectors = index.ntotal
vectors = np.array([index.reconstruct(i) for i in range(num_vectors)])

# ==========================================
# 3. Parse Labels to Histograms
# ==========================================
signatures = []
valid_mask = [] # To track which models have valid labels

for meta in tqdm(metadata, desc="Parsing feature histograms"):
    base_name = meta.replace('.step', '')
    label_path = LABEL_DIR / f"{base_name}.face_truth"
    
    if label_path.exists():
        with open(label_path, 'rb') as file:
            face_labels = pickle.load(file)
            
        # Handle both dict {face_id: label} and list [label, label, ...] formats
        labels = face_labels.values() if isinstance(face_labels, dict) else face_labels
        
        # Create a 16-D frequency histogram
        hist = np.zeros(NUM_CLASSES)
        for lbl in labels:
            if 0 <= lbl < NUM_CLASSES:
                hist[lbl] += 1
                
        signatures.append(hist)
        valid_mask.append(True)
    else:
        signatures.append(np.zeros(NUM_CLASSES))
        valid_mask.append(False)

signatures = np.array(signatures)
print(f"Loaded {sum(valid_mask)} valid label signatures out of {num_vectors} models.")

Parsing feature histograms:   0%|          | 0/15488 [00:00<?, ?it/s]

Loaded 15488 valid label signatures out of 15488 models.


## Retrieval Evaluation
Compute Recall@K and MRR. A retrieved model is relevant if the cosine similarity between its histogram and the query's histogram is $\ge$ `MATCH_THRESHOLD`.

In [3]:
# ==========================================
# 4. Perform Retrieval & Metrics Calculation
# ==========================================
def cosine_sim(v1, v2):
    norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

k_max = 10
distances, indices = index.search(vectors, k_max + 1)

recalls = {1: 0, 5: 0, 10: 0}
mrr = 0.0
valid_queries = 0

for i in tqdm(range(num_vectors), desc="Evaluating Retrieval"):
    if not valid_mask[i]:
        continue # Skip queries without ground truth
        
    query_sig = signatures[i]
    if np.linalg.norm(query_sig) == 0:
        continue # Skip if the model has no recognized features
        
    valid_queries += 1
    retrieved_idx = indices[i][1:] # Exclude the query itself (index 0)
    
    # Check relevance for Top-K
    is_relevant = []
    for idx in retrieved_idx:
        if not valid_mask[idx]:
            is_relevant.append(False)
        else:
            sim = cosine_sim(query_sig, signatures[idx])
            is_relevant.append(sim >= MATCH_THRESHOLD)
            
    # Calculate Recall@K
    for k in recalls.keys():
        if any(is_relevant[:k]):
            recalls[k] += 1
            
    # Calculate MRR
    for rank, relevant in enumerate(is_relevant):
        if relevant:
            mrr += 1.0 / (rank + 1)
            break

# Normalize metrics
for k in recalls.keys():
    recalls[k] /= valid_queries
mrr /= valid_queries

print("\n" + "="*45)
print("🏆 MANUFACTURING-AWARE EVALUATION 🏆")
print("="*45)
print(f"Match Threshold (Cosine Sim): >= {MATCH_THRESHOLD}")
print(f"Total Valid Queries: {valid_queries}")
print("-" * 45)
print(f"Recall@1:  {recalls[1]*100:>6.2f}%")
print(f"Recall@5:  {recalls[5]*100:>6.2f}%")
print(f"Recall@10: {recalls[10]*100:>6.2f}%")
print(f"MRR:       {mrr:>8.4f}")
print("="*45)

Evaluating Retrieval:   0%|          | 0/15488 [00:00<?, ?it/s]


🏆 MANUFACTURING-AWARE EVALUATION 🏆
Match Threshold (Cosine Sim): >= 0.95
Total Valid Queries: 15488
---------------------------------------------
Recall@1:   12.74%
Recall@5:   32.73%
Recall@10:  44.25%
MRR:         0.2130
